# Assignment 2: Advanced RAG Techniques
## Day 6 Session 2 - Advanced RAG Fundamentals

**OBJECTIVE:** Implement advanced RAG techniques including postprocessors, response synthesizers, and structured outputs.

**LEARNING GOALS:**
- Understand and implement node postprocessors for filtering and reranking
- Learn different response synthesis strategies (TreeSummarize, Refine)
- Create structured outputs using Pydantic models
- Build advanced retrieval pipelines with multiple processing stages

**DATASET:** Use the same data folder as Assignment 1 (`Day_6/session_2/data/`)

**PREREQUISITES:** Complete Assignment 1 first

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it
3. The answers can be found in the `03_advanced_rag_techniques.ipynb` notebook
4. Each technique builds on the previous one


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q uv
!uv pip install --system -r /content/drive/MyDrive/outskill_c4/requirements.txt

In [1]:
import os
from getpass import getpass

# securely input your key
os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter key")
print("✓ OpenRouter key set successfully")

✓ OpenRouter key set successfully


In [2]:
# Import required libraries for advanced RAG
import os
from pathlib import Path
from typing import Dict, List, Optional, Any
from pydantic import BaseModel, Field

# Core LlamaIndex components
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever

# Vector store
from llama_index.vector_stores.lancedb import LanceDBVectorStore

# Embeddings and LLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter

# Advanced RAG components (we'll use these in the assignments)
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.response_synthesizers import TreeSummarize, Refine, CompactAndRefine
from llama_index.core.output_parsers import PydanticOutputParser

print("Advanced RAG libraries imported successfully!")


/Users/sumisama/Desktop/sumit/outskill/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Advanced RAG libraries imported successfully!


In [3]:
# Configure Advanced RAG Settings (Using OpenRouter)
def setup_advanced_rag_settings():
    """
    Configure LlamaIndex with optimized settings for advanced RAG.
    Uses local embeddings and OpenRouter for LLM operations.
    """
    # Check for OpenRouter API key
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        print("OPENROUTER_API_KEY not found - LLM operations will be limited")
        print("You can still complete postprocessor and retrieval exercises")
    else:
        print("OPENROUTER_API_KEY found - full advanced RAG functionality available")

        # Configure OpenRouter LLM
        Settings.llm = OpenRouter(
            api_key=api_key,
            model="gpt-4o",
            temperature=0.1  # Lower temperature for more consistent responses
        )

    # Configure local embeddings (no API key required)
    Settings.embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        trust_remote_code=True
    )

    # Advanced RAG configuration
    Settings.chunk_size = 512  # Smaller chunks for better precision
    Settings.chunk_overlap = 50

    print("Advanced RAG settings configured")
    print("- Chunk size: 512 (optimized for precision)")
    print("- Using local embeddings for cost efficiency")
    print("- OpenRouter LLM ready for response synthesis")

# Setup the configuration
setup_advanced_rag_settings()


OPENROUTER_API_KEY found - full advanced RAG functionality available
Advanced RAG settings configured
- Chunk size: 512 (optimized for precision)
- Using local embeddings for cost efficiency
- OpenRouter LLM ready for response synthesis


In [6]:
# Setup: Create index from Assignment 1 (reuse the basic functionality)
def setup_basic_index(data_folder: str, force_rebuild: bool = False):
    """
    Create a basic vector index that we'll enhance with advanced techniques.
    This reuses the concepts from Assignment 1.
    """
    # Create vector store
    vector_store = LanceDBVectorStore(
        uri="/Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/storage/assignment_multimodal_vectordb2",
        table_name="documents"
    )

    # Load documents
    if not Path(data_folder).exists():
        print(f"Data folder not found: {data_folder}")
        return None

    reader = SimpleDirectoryReader(input_dir=data_folder, recursive=True)
    documents = reader.load_data()

    # Create storage context and index
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )

    print(f"Basic index created with {len(documents)} documents")
    print("Ready for advanced RAG techniques!")
    return index

data_folder_path = "/Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/data"
# Create the basic index
print("Setting up basic index for advanced RAG...")
index = setup_basic_index(data_folder=data_folder_path)

if index:
    print("Ready to implement advanced RAG techniques!")
else:
    print("Failed to create index - check data folder path")


Table documents doesn't exist yet. Please add some data to create it.


Setting up basic index for advanced RAG...


/Users/sumisama/Desktop/sumit/outskill/.venv/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/sumisama/Desktop/sumit/outskill/.venv/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/sumisama/Desktop/sumit/outskill/.venv/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
Generating embeddings: 100%|██████████| 94/94 [00:03<00:00, 25.08it/s]
2026-05-03 11:45:30,913 - INFO - Create new table documents adding data.


Basic index created with 42 documents
Ready for advanced RAG techniques!
Ready to implement advanced RAG techniques!


[2026-05-03T06:15:30Z WARN  lance::dataset::write::insert] No existing dataset at /Users/sumisama/Desktop/sumit/outskill/ai-accelerator-C6/Day 6/session_2/storage/assignment_multimodal_vectordb2/documents.lance, it will be created


## 1. Node Postprocessors - Similarity Filtering

**Concept:** Postprocessors refine retrieval results after the initial vector search. The `SimilarityPostprocessor` filters out chunks that fall below a relevance threshold.

**Why it matters:** Raw vector search often returns some irrelevant results. Filtering improves precision and response quality.

Complete the function below to create a query engine with similarity filtering.


In [7]:
def create_query_engine_with_similarity_filter(index, similarity_cutoff: float = 0.3, top_k: int = 10):
    """
    Create a query engine that filters results based on similarity scores.

    TODO: Complete this function to create a query engine with similarity postprocessing.


    Args:
        index: Vector index to query
        similarity_cutoff: Minimum similarity score (0.0 to 1.0)
        top_k: Number of initial results to retrieve before filtering

    Returns:
        Query engine with similarity filtering
    """
    # HINT: Use index.as_query_engine() with node_postprocessors parameter containing SimilarityPostprocessor
    # TODO: Create similarity postprocessor with the cutoff threshold
    similarity_processor = SimilarityPostprocessor(similarity_cutoff=similarity_cutoff)

    # TODO: Create query engine with similarity filtering
    query_engine = index.as_query_engine(
        node_postprocessors=[similarity_processor],
        similarity_top_k=top_k
    )

    # return query_engine
    return query_engine

# Test the function
if index:
    filtered_engine = create_query_engine_with_similarity_filter(index, similarity_cutoff=0.3)

    if filtered_engine:
        print("Query engine with similarity filtering created")

        # Test query
        test_query = "What are the benefits of AI agents?"
        print(f"\nTesting query: '{test_query}'")

        # Uncomment when implemented:
        response = filtered_engine.query(test_query)
        print(f"Response: {response}")
        print(f"Filtered Response: {response}")
    else:
        print("Failed to create filtered query engine")
else:
    print("No index available - run previous cells first")

Query engine with similarity filtering created

Testing query: 'What are the benefits of AI agents?'


2026-05-03 11:46:47,990 - INFO - query_type :, vector
2026-05-03 11:46:49,474 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-03 11:46:51,216 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-03 11:46:52,434 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Response: AI agents offer significant benefits, including the ability to understand, reason, and act on their environment, which allows them to solve complex problems effectively. They can autonomously make decisions and interact with their surroundings, supporting humans in various tasks. Additionally, multi-agent systems enhance performance by providing diverse perspectives and enabling parallel task execution, which is particularly beneficial in collaborative scenarios requiring multiple strategies.
Filtered Response: AI agents offer significant benefits, including the ability to understand, reason, and act on their environment, which allows them to solve complex problems effectively. They can autonomously make decisions and interact with their surroundings, supporting humans in various tasks. Additionally, multi-agent systems enhance performance by providing diverse perspectives and enabling parallel task execution, which is particularly beneficial in collaborative scenarios requir

## 2. Structured Outputs with Pydantic Models

**Concept:** Structured outputs ensure predictable, parseable responses using Pydantic models. This is essential for API endpoints and data pipelines.

**Why it matters:** Instead of free-text responses, you get type-safe, validated data structures that applications can reliably process.

Complete the function below to create a structured output system for extracting research paper information.


In [14]:
# First, define the Pydantic models for structured outputs
class ResearchPaperInfo(BaseModel):
    """Structured information about a research paper or AI concept."""
    title: str = Field(description="The main title or concept name")
    key_points: List[str] = Field(description="3-5 main points or findings")
    applications: List[str] = Field(description="Practical applications or use cases")
    summary: str = Field(description="Brief 2-3 sentence summary")


# Import the missing component
from llama_index.core.program import LLMTextCompletionProgram

def create_structured_output_program(output_model: BaseModel = ResearchPaperInfo):
    """
    Create a structured output program using Pydantic models.

    TODO: Complete this function to create a structured output program.

    Args:
        output_model: Pydantic model class for structured output

    Returns:
        LLMTextCompletionProgram that returns structured data
    """
    # TODO: Create output parser with the Pydantic model (HINT: Use PydanticOutputParser)
    output_parser = PydanticOutputParser(output_cls=output_model)

    # TODO: Create the structured output program (HINT output_parser and prompt_template_str will be passed to the function)
    program = LLMTextCompletionProgram.from_defaults(output_parser=output_parser, prompt_template_str="Given the following context, extract structured information about the main topic in JSON format according to the specified schema. Ensure that each field on the schema is printed on a new line '\n' for better readability.")

    # return program
    return program

    # PLACEHOLDER - Replace with actual implementation
    # print(f"TODO: Create structured output program with {output_model.__name__}")
    # return None

# Test the function
if index:
    structured_program = create_structured_output_program(ResearchPaperInfo)

    if structured_program:
        print("Structured output program created")

        # Test with retrieval and structured extraction
        structure_query = "Tell me about AI agents and their capabilities"
        print(f"Testing structured query: '{structure_query}'")

        # Get context for structured extraction (Uncomment when implemented)
        retriever = VectorIndexRetriever(index=index, similarity_top_k=3)
        nodes = retriever.retrieve(structure_query)
        context = "\n".join([node.text for node in nodes])

        # Uncomment when implemented:
        response = structured_program(context=context, query=structure_query)
        print(f"Structured Response:\n{response}")
        # print("   (Complete the function above to get structured JSON output)")

        # print("Expected output format:")
        # print("   - title: String")
        # print("   - key_points: List of strings")
        # print("   - applications: List of strings")
        # print("   - summary: String")
    else:
        print("Failed to create structured output program")
else:
    print("No index available - run previous cells first")


2026-05-03 12:05:31,184 - INFO - query_type :, vector


Structured output program created
Testing structured query: 'Tell me about AI agents and their capabilities'


2026-05-03 12:05:32,166 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Structured Response:
title='Advancements in Neural Network Optimization' key_points=['Introduction of a novel optimization algorithm that reduces training time by 30%.', 'Improved accuracy in image classification tasks by 5% compared to existing methods.', 'Demonstrated scalability across various neural network architectures.', 'Enhanced robustness to overfitting through adaptive learning rate adjustments.'] applications=['Image and speech recognition systems.', 'Autonomous vehicle navigation.', 'Real-time data analysis in financial markets.', 'Healthcare diagnostics and predictive modeling.'] summary='This research paper presents a new optimization algorithm for neural networks that significantly reduces training time and improves accuracy. The algorithm is versatile, showing effectiveness across different architectures and applications, while also enhancing robustness against overfitting.'
